# Chapter 05: Functional Analysis

**Source span:** PDF pages 51-54 of *Mathematical Foundations of Geometric Deep Learning*.

**Chapter goal:** make the analytic words in the source span executable: Cauchy tails, completeness, Banach and Hilbert geometry, orthonormal function coordinates, operators, adjoints, compactness, and scalar-valued functionals.

**Chapter question:** what changes when vectors become functions and linear algebra has to respect limits?

Functional analysis is where geometric deep learning stops treating vectors as only finite arrays. A signal on a graph, a field on a manifold, or a function on a domain can still be added, scaled, projected, and transformed, but convergence now matters. A method that is stable in finite dimensions can fail when a sequence converges to a point outside the space being used. A basis expansion can be correct only after we specify the norm in which the reconstruction converges. An operator can be more than a matrix: it can be a rule on functions whose boundedness, adjoint, or compactness controls whether spectral methods are well behaved.

This notebook is standalone. It uses the source pages for concept coverage and terminology, but the prose, computations, and figures are original. Every artifact is generated under `artifacts/chapter-05/` and is paired with an invariant or sanity check.

## Translation guide

| Source concept | Computational object in this notebook | What to inspect |
| --- | --- | --- |
| Cauchy sequence | tail diameters of sampled sequences | tail points become mutually close even before a limit is named |
| Completeness | whether the limiting point belongs to the declared space | a boundary point or irrational limit can be outside the space |
| Banach space | a complete normed space, visualized through norm balls | norms define distances; not every norm comes from angles |
| Hilbert space | a complete inner-product space | projections, orthogonality, and the parallelogram law become available |
| Orthonormal basis | sampled Fourier-style basis functions in `L^2([0,1])` | coefficients are inner products and reconstruction is an orthogonal projection |
| Operator | a matrix or diagonal rule acting on vectors/functions | boundedness, isometry, projection, adjoint, rank, compactness |
| Functional | a scalar-valued probe `phi(f)` | in Hilbert space, many probes look like inner products with a representing function |

### Library routing

- **NumPy** carries finite approximations to sequence tails, norms, bases, operators, and inner products.
- **Matplotlib** is used for durable 2D diagrams: boundary gaps, norm balls, operator action, weak/strong probes, and functional products.
- **Plotly** is used for the partial-sum reconstruction because the learner should switch the number of basis terms and inspect convergence dynamically.
- **SymPy** checks the adjoint identity symbolically in the finite-dimensional model.
- **NetworkX** draws the dependency graph linking completeness, Hilbert geometry, basis coordinates, operators, adjoints, and dual functionals.
- **Pandas/CSV/JSON** record reconstruction and denoising tables plus invariant summaries that the final sanity cell asserts.

## Visual storyboard

| Step | Artifact | Representation | Invariant or check |
| --- | --- | --- | --- |
| Completeness gap at a boundary | `figures/cauchy-completeness-boundary.png` | number-line tails in `(0,1]` versus `(0,1)` | tail diameter shrinks while the open interval misses the limit |
| Rational Cauchy gap | `figures/rational-cauchy-sqrt2-completion-gap.png` | exact rational Newton iterates | residual to `sqrt(2)` decreases while the limit is not rational |
| Banach/Hilbert geometry | `figures/banach-hilbert-norm-projection-geometry.png` | unit balls and orthogonal projection | `L^2` parallelogram identity holds; `L^1` version fails |
| Function coordinates | `plots/l2-orthonormal-reconstruction-error.png` and `html/orthonormal-reconstruction-partial-sums.html` | sampled orthonormal functions and partial sums | Gram matrix near identity; Parseval energy balance; reconstruction error shrinks |
| Operators and adjoints | `figures/operator-adjoint-property-gallery.png` | images of the unit circle under several maps | adjoint identity, isometry, projection, and non-self-adjoint checks |
| Weak/strong/compact behavior | `figures/weak-strong-compact-probes.png` | sequence probes in `ell^2` | weak probes vanish while norms do not; compact diagonal images vanish strongly |
| Functionals | `figures/functional-riesz-representer-probe.png` | inner-product probe and representing function | linearity and Cauchy-Schwarz bound |
| Proof scaffold | `figures/functional-analysis-dependency-graph.png` | directed dependency graph | graph is acyclic and contains the chapter's conceptual paths |
| Applied lab | `figures/orthogonal-projection-denoising-lab.png` | low-frequency orthogonal projection | projected signal beats the noisy signal in `L^2` error |

In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the MFGDL book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = "chapter-05"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
for kind in ["figures", "html", "plots", "tables", "checks", "data"]:
    (ARTIFACT_ROOT / kind).mkdir(parents=True, exist_ok=True)

artifact_paths = []
checks = {"source_span": "PDF pages 51-54", "chapter_goal": "functional analysis as executable geometry for GDL"}

def remember(path):
    path = Path(path)
    if path not in artifact_paths:
        artifact_paths.append(path)
    return path

BOOK_ROOT

In [ ]:
import json
import math
from fractions import Fraction

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from IPython.display import Markdown, display

from utils.artifacts import display_artifact, save_json, save_matplotlib, save_plotly_html, save_table_csv
from utils.notebook_checks import assert_chapter_artifacts, assert_nonblank_image
from utils.plotting import PALETTE, add_note, arrow2, close, style_axis

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white"})

## 1. Cauchy Sequences And Completeness Gaps

A Cauchy sequence is a promise about tails: after some index, all later elements are mutually close. Completeness is the separate promise that the tail has somewhere inside the space to land.

The first visual uses `d_n = 1 - 1/n`, starting at `n=2`. The same points live in `(0,1]` and `(0,1)`, and their tail diameters shrink. The difference is the endpoint: `1` is part of `(0,1]` but not part of `(0,1)`. The second visual uses exact rational Newton iterates for `sqrt(2)`. The sequence remains rational at every finite step, becomes Cauchy in the absolute-value norm, and converges in the real completion rather than in `Q`.

In [ ]:
n = np.arange(2, 90)
d = 1.0 - 1.0 / n
tail_starts = np.arange(2, 80)
tail_diameters = np.array([d[n >= start].max() - d[n >= start].min() for start in tail_starts])

fig, axes = plt.subplots(2, 1, figsize=(9, 5.8), height_ratios=[1.25, 1.0])
ax = axes[0]
for y, label, endpoint_marker in [(1.0, "V1 = (0, 1]", "closed"), (0.0, "V2 = (0, 1)", "open")]:
    ax.hlines(y, 0, 1, color="#cbd5e1", linewidth=8, alpha=0.75)
    ax.scatter(d, np.full_like(d, y), s=np.linspace(18, 70, d.size), color=PALETTE["blue"], alpha=0.55, edgecolor="white", linewidth=0.3)
    ax.scatter([0], [y], s=120, facecolors="white", edgecolors=PALETTE["gray"], linewidth=2)
    if endpoint_marker == "closed":
        ax.scatter([1], [y], s=120, color=PALETTE["green"], edgecolors="white", linewidth=1.5)
    else:
        ax.scatter([1], [y], s=120, facecolors="white", edgecolors=PALETTE["red"], linewidth=2)
    ax.text(-0.04, y, label, ha="right", va="center", color=PALETTE["ink"])
ax.axvline(1, color=PALETTE["red"], linestyle="--", linewidth=1.4, label="tail target 1")
ax.set_yticks([])
ax.set_xlim(-0.05, 1.05)
style_axis(ax, "A Cauchy tail can point at a missing boundary", xlabel="number-line position")
ax.legend(loc="lower right")
ax = axes[1]
ax.semilogy(tail_starts, tail_diameters, color=PALETTE["teal"], linewidth=2)
ax.scatter(tail_starts[::8], tail_diameters[::8], color=PALETTE["teal"], s=28)
style_axis(ax, "Tail diameter shrinks even before completeness is known", xlabel="tail starts after N", ylabel="max tail distance")
fig.tight_layout()
boundary_path = remember(save_matplotlib(fig, TOPIC, "cauchy-completeness-boundary.png", root=BOOK_ROOT / "artifacts"))
close(fig)

rational_iterates = []
xq = Fraction(1, 1)
for index in range(8):
    xq = (xq + Fraction(2, 1) / xq) / 2
    rational_iterates.append(xq)
rat_values = np.array([float(value) for value in rational_iterates])
rat_residuals = np.array([abs(float(value * value - 2)) for value in rational_iterates])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(np.arange(1, len(rat_values) + 1), rat_values, "o-", color=PALETTE["green"], label="rational iterate")
axes[0].axhline(math.sqrt(2), color=PALETTE["red"], linestyle="--", label="real limit sqrt(2)")
style_axis(axes[0], "Rational iterates approach an irrational limit", xlabel="iteration", ylabel="value")
axes[0].legend(loc="best", fontsize=8)
axes[1].semilogy(np.arange(1, len(rat_residuals) + 1), rat_residuals, "o-", color=PALETTE["violet"])
style_axis(axes[1], "Residual x_n^2 - 2 collapses", xlabel="iteration", ylabel="absolute residual")
fig.tight_layout()
rational_path = remember(save_matplotlib(fig, TOPIC, "rational-cauchy-sqrt2-completion-gap.png", root=BOOK_ROOT / "artifacts"))
close(fig)

checks["cauchy_completeness"] = {
    "boundary_tail_diameter_last10": float(d[-10:].max() - d[-10:].min()),
    "open_interval_contains_limit": bool(0 < 1 < 1),
    "closed_right_interval_contains_limit": bool(0 < 1 <= 1),
    "sqrt2_last_rational": f"{rational_iterates[-1].numerator}/{rational_iterates[-1].denominator}",
    "sqrt2_residual_last": float(rat_residuals[-1]),
    "sqrt2_residual_decreased": bool(rat_residuals[-1] < rat_residuals[0]),
}

display_artifact(boundary_path, width=760)
display_artifact(rational_path, width=760)
checks["cauchy_completeness"]

## 2. Banach Geometry Versus Hilbert Geometry

A Banach space needs a complete norm. A Hilbert space needs more: an inner product that induces the norm. The extra structure is visible. Inner products give angles, orthogonal projections, and the parallelogram identity. A norm such as `L^1` still measures length, but it does not behave like squared Euclidean length and does not define orthogonal projection by itself.

The visual below compares unit balls and then draws one Hilbert-space projection. The residual vector is perpendicular to the subspace, which is the geometric reason least-squares projection works.

In [ ]:
def lp_norm(vector, p):
    vector = np.asarray(vector, dtype=float)
    if p == np.inf:
        return float(np.max(np.abs(vector)))
    return float(np.sum(np.abs(vector) ** p) ** (1.0 / p))

u = np.array([1.0, 0.0])
v = np.array([0.0, 0.8])
parallelogram_l2 = lp_norm(u + v, 2) ** 2 + lp_norm(u - v, 2) ** 2 - 2 * lp_norm(u, 2) ** 2 - 2 * lp_norm(v, 2) ** 2
parallelogram_l1 = lp_norm(u + v, 1) ** 2 + lp_norm(u - v, 1) ** 2 - 2 * lp_norm(u, 1) ** 2 - 2 * lp_norm(v, 1) ** 2
subspace_direction = np.array([1.0, 0.28])
e = subspace_direction / np.linalg.norm(subspace_direction)
f_vec = np.array([0.55, 1.15])
projection = np.dot(f_vec, e) * e
residual = f_vec - projection
orthogonality_residual = float(np.dot(residual, e))

fig, axes = plt.subplots(2, 2, figsize=(9, 8))
t = np.linspace(0, 2 * np.pi, 400)
axes[0, 0].plot(np.cos(t), np.sin(t), color=PALETTE["blue"], label="L2")
axes[0, 0].plot([1, 0, -1, 0, 1], [0, 1, 0, -1, 0], color=PALETTE["gold"], label="L1")
axes[0, 0].plot([1, 1, -1, -1, 1], [1, -1, -1, 1, 1], color=PALETTE["teal"], label="L-infinity")
style_axis(axes[0, 0], "Different norms, different unit balls", equal=True)
axes[0, 0].legend(fontsize=8, loc="upper right")
axes[0, 0].set_xlim(-1.25, 1.25); axes[0, 0].set_ylim(-1.25, 1.25)
origin = np.zeros(2)
for vector, color, label in [(u, PALETTE["blue"], "u"), (v, PALETTE["green"], "v"), (u + v, PALETTE["violet"], "u+v"), (u - v, PALETTE["red"], "u-v")]:
    arrow2(axes[0, 1], origin, vector, color=color, label=label)
style_axis(axes[0, 1], "Parallelogram identity detects Hilbert geometry", equal=True)
axes[0, 1].set_xlim(-1.4, 1.4); axes[0, 1].set_ylim(-1.0, 1.7)
add_note(axes[0, 1], f"L2 residual {parallelogram_l2:.1e}\nL1 residual {parallelogram_l1:.2f}")
line = np.outer(np.linspace(-1.2, 1.4, 2), e)
axes[1, 0].plot(line[:, 0], line[:, 1], color=PALETTE["gray"], linewidth=3, label="1D subspace")
arrow2(axes[1, 0], origin, f_vec, color=PALETTE["blue"], label="f")
arrow2(axes[1, 0], origin, projection, color=PALETTE["green"], label="proj f")
arrow2(axes[1, 0], projection, residual, color=PALETTE["red"], label="residual")
style_axis(axes[1, 0], "Hilbert projection: residual is orthogonal", equal=True)
axes[1, 0].legend(fontsize=8, loc="upper left")
axes[1, 0].set_xlim(-0.4, 1.1); axes[1, 0].set_ylim(-0.15, 1.35)
add_note(axes[1, 0], f"<residual, e> = {orthogonality_residual:.1e}")
grid = np.linspace(-1.2, 1.2, 240)
X, Y = np.meshgrid(grid, grid)
axes[1, 1].contour(X, Y, np.abs(X) + np.abs(Y), levels=[0.5, 1.0, 1.5], colors=PALETTE["gold"], linewidths=1.4)
axes[1, 1].contour(X, Y, np.sqrt(X * X + Y * Y), levels=[0.5, 1.0, 1.5], colors=PALETTE["blue"], linewidths=1.4)
style_axis(axes[1, 1], "Same vectors, different distance geometry", equal=True)
axes[1, 1].set_xlim(-1.2, 1.2); axes[1, 1].set_ylim(-1.2, 1.2)
add_note(axes[1, 1], "blue: L2 contours\ngold: L1 contours")
fig.tight_layout()
banach_hilbert_path = remember(save_matplotlib(fig, TOPIC, "banach-hilbert-norm-projection-geometry.png", root=BOOK_ROOT / "artifacts"))
close(fig)
checks["banach_hilbert_geometry"] = {
    "parallelogram_l2_residual": float(abs(parallelogram_l2)),
    "parallelogram_l1_residual": float(abs(parallelogram_l1)),
    "projection_residual_dot_subspace": float(abs(orthogonality_residual)),
}
display_artifact(banach_hilbert_path, width=760)
checks["banach_hilbert_geometry"]

## 3. Functions As Infinite-Dimensional Vectors

The source span treats square-integrable functions as vectors whose coordinates are coefficients against an orthonormal basis. In code we cannot store an infinite basis, so we sample a periodic domain and use a finite Fourier-style basis. This is still enough to see the principle:

1. The Gram matrix of an orthonormal basis should look like the identity.
2. A coefficient is an inner product.
3. Adding more orthonormal terms gives the best projection inside the chosen span.
4. Parseval's identity says energy in the function equals energy in its coefficients when the basis spans the signal.

The interactive Plotly artifact lets you move through partial sums. The static plot records coefficients and error decay for auditability.

In [ ]:
sample_count = 768
x_grid = np.linspace(0.0, 1.0, sample_count, endpoint=False)
dx = 1.0 / sample_count
basis_names = ["1"]
basis_vectors = [np.ones_like(x_grid)]
for k in range(1, 7):
    basis_names.append(f"sqrt2 sin {k}")
    basis_vectors.append(np.sqrt(2.0) * np.sin(2 * np.pi * k * x_grid))
    basis_names.append(f"sqrt2 cos {k}")
    basis_vectors.append(np.sqrt(2.0) * np.cos(2 * np.pi * k * x_grid))
B = np.vstack(basis_vectors)
f = 1.10 + 0.72 * np.sin(2 * np.pi * x_grid) - 0.45 * np.cos(4 * np.pi * x_grid) + 0.32 * np.sin(6 * np.pi * x_grid) + 0.18 * np.cos(10 * np.pi * x_grid)
coefficients = B @ f * dx
gram = B @ B.T * dx
energy_f = float(np.sum(f * f) * dx)
energy_coefficients = float(np.sum(coefficients * coefficients))
partial_counts = [1, 2, 3, 5, 7, 9, 11, len(basis_names)]
reconstructions = {}
rows = []
for count in partial_counts:
    recon = coefficients[:count] @ B[:count]
    reconstructions[count] = recon
    error = float(np.sqrt(np.sum((f - recon) ** 2) * dx))
    captured = float(np.sum(coefficients[:count] ** 2) / energy_f)
    rows.append({"basis_terms": count, "l2_error": error, "captured_energy_fraction": captured})
reconstruction_table_path = remember(save_table_csv(rows, TOPIC, "reconstruction-error.csv", root=BOOK_ROOT / "artifacts"))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
axes[0].plot(x_grid, f, color=PALETTE["ink"], linewidth=2, label="target f")
for count, color in [(1, PALETTE["gray"]), (3, PALETTE["gold"]), (7, PALETTE["teal"]), (len(basis_names), PALETTE["blue"] )]:
    axes[0].plot(x_grid, reconstructions[count], color=color, linewidth=1.6, alpha=0.85, label=f"{count} terms")
style_axis(axes[0], "Partial sums converge in L2", xlabel="x", ylabel="value")
axes[0].legend(fontsize=7, loc="best")
bar_positions = np.arange(len(basis_names))
axes[1].bar(bar_positions, coefficients, color=PALETTE["violet"])
axes[1].set_xticks(bar_positions)
axes[1].set_xticklabels(basis_names, rotation=70, ha="right", fontsize=7)
style_axis(axes[1], "Coordinates are inner products", ylabel="coefficient")
axes[2].plot([row["basis_terms"] for row in rows], [row["l2_error"] for row in rows], "o-", color=PALETTE["red"])
style_axis(axes[2], "Projection error by basis size", xlabel="basis terms", ylabel="L2 error")
fig.tight_layout()
reconstruction_png_path = remember(save_matplotlib(fig, TOPIC, "l2-orthonormal-reconstruction-error.png", kind="plots", root=BOOK_ROOT / "artifacts"))
close(fig)

plotly_fig = go.Figure()
plotly_fig.add_trace(go.Scatter(x=x_grid, y=f, mode="lines", name="target f", line={"color": "#1f2933", "width": 3}))
for index, count in enumerate(partial_counts):
    plotly_fig.add_trace(go.Scatter(x=x_grid, y=reconstructions[count], mode="lines", name=f"{count} terms", visible=(index == 0), line={"width": 2}))
steps = []
for index, count in enumerate(partial_counts):
    visible = [True] + [False] * len(partial_counts)
    visible[index + 1] = True
    steps.append({"method": "update", "label": str(count), "args": [{"visible": visible}, {"title": f"Orthonormal reconstruction with {count} basis terms"}]})
plotly_fig.update_layout(title="Orthonormal reconstruction with 1 basis term", height=520, xaxis_title="x in [0,1)", yaxis_title="function value", sliders=[{"active": 0, "currentvalue": {"prefix": "basis terms: "}, "steps": steps}], template="plotly_white")
reconstruction_html_path = remember(save_plotly_html(plotly_fig, TOPIC, "orthonormal-reconstruction-partial-sums.html", root=BOOK_ROOT / "artifacts"))

checks["orthonormal_reconstruction"] = {
    "basis_count": int(len(basis_names)),
    "gram_max_abs_error": float(np.max(np.abs(gram - np.eye(len(basis_names))))),
    "parseval_residual": float(abs(energy_f - energy_coefficients)),
    "full_reconstruction_l2_error": float(rows[-1]["l2_error"]),
    "first_reconstruction_l2_error": float(rows[0]["l2_error"]),
}
display_artifact(reconstruction_png_path, width=900)
display_artifact(reconstruction_html_path, width="100%", height=560)
pd.DataFrame(rows)

## 4. Operators And Adjoints

An operator is a map between spaces. In a finite-dimensional Hilbert model, operators are matrices, boundedness is controlled by singular values, and the adjoint is the transpose for the real inner product. These small matrices are not the full theory, but they make the definitions inspectable.

The panels below show how several operators move the unit circle: a rotation is an isometry, a diagonal scaling is bounded but not length-preserving, an orthogonal projection is self-adjoint and idempotent, and a shear is linear and bounded but not self-adjoint. SymPy then verifies the adjoint identity symbolically for a general `2 x 2` real matrix.

In [ ]:
theta = np.pi / 5
rotation = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
scaling = np.array([[1.45, 0.0], [0.0, 0.65]])
proj_dir = np.array([1.0, 0.55])
proj_dir = proj_dir / np.linalg.norm(proj_dir)
projection_matrix = np.outer(proj_dir, proj_dir)
shear = np.array([[1.0, 0.85], [0.0, 1.0]])
operators = [(rotation, "isometry rotation", PALETTE["green"]), (scaling, "bounded scaling", PALETTE["blue"]), (projection_matrix, "self-adjoint projection", PALETTE["gold"]), (shear, "non-self-adjoint shear", PALETTE["red"])]

circle_t = np.linspace(0, 2 * np.pi, 400)
unit_circle = np.c_[np.cos(circle_t), np.sin(circle_t)]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, (matrix, title, color) in zip(axes, operators):
    image = unit_circle @ matrix.T
    ax.plot(unit_circle[:, 0], unit_circle[:, 1], color="#cbd5e1", linewidth=1.4, label="unit circle")
    ax.plot(image[:, 0], image[:, 1], color=color, linewidth=2.2, label="A circle")
    for vector, label in [(np.array([1.0, 0.0]), "e1"), (np.array([0.0, 1.0]), "e2")]:
        arrow2(ax, np.zeros(2), matrix @ vector, color=PALETTE["ink"], label=label)
    style_axis(ax, title, equal=True)
    ax.set_xlim(-1.8, 1.8); ax.set_ylim(-1.8, 1.8)
    ax.legend(fontsize=7, loc="lower left")
fig.tight_layout()
operator_path = remember(save_matplotlib(fig, TOPIC, "operator-adjoint-property-gallery.png", root=BOOK_ROOT / "artifacts"))
close(fig)

rng = np.random.default_rng(5)
A_numeric = np.array([[1.2, -0.4], [0.35, 0.9]])
adjoint_residuals = []
for _ in range(20):
    left_u = rng.normal(size=2)
    right_v = rng.normal(size=2)
    adjoint_residuals.append(abs(np.dot(A_numeric @ left_u, right_v) - np.dot(left_u, A_numeric.T @ right_v)))
A11, A12, A21, A22, u1, u2, v1, v2 = sp.symbols("A11 A12 A21 A22 u1 u2 v1 v2")
A_sym = sp.Matrix([[A11, A12], [A21, A22]])
u_sym = sp.Matrix([u1, u2])
v_sym = sp.Matrix([v1, v2])
symbolic_adjoint_identity = sp.simplify((A_sym * u_sym).dot(v_sym) - u_sym.dot(A_sym.T * v_sym))

checks["operators_adjoint"] = {
    "rotation_length_residual": float(np.max(np.abs(np.linalg.norm(unit_circle @ rotation.T, axis=1) - 1.0))),
    "scaling_operator_norm": float(np.linalg.svd(scaling, compute_uv=False)[0]),
    "projection_idempotent_residual": float(np.max(np.abs(projection_matrix @ projection_matrix - projection_matrix))),
    "projection_self_adjoint_residual": float(np.max(np.abs(projection_matrix.T - projection_matrix))),
    "shear_self_adjoint_residual": float(np.max(np.abs(shear.T - shear))),
    "numeric_adjoint_max_residual": float(max(adjoint_residuals)),
    "symbolic_adjoint_identity_zero": bool(symbolic_adjoint_identity == 0),
}
display_artifact(operator_path, width=900)
checks["operators_adjoint"]

## 5. Weak Limits, Strong Limits, And Compact Operators

The source span distinguishes weak and strong convergence because compact operators are often defined by how they convert weak behavior into norm behavior. The clean sequence to remember in `ell^2` is the standard basis sequence `e_n`.

- `||e_n|| = 1`, so `e_n` does **not** converge strongly to zero.
- For any fixed square-summable probe `w`, the coordinate probe `<e_n, w> = w_n` tends to zero, so the sequence converges weakly to zero.
- A compact diagonal operator with singular values `1/n` sends `e_n` to `(1/n)e_n`, whose norm tends to zero.

The plot uses finite prefixes to show those three diagnostics together.

In [ ]:
indices = np.arange(1, 101)
strong_norms = np.ones_like(indices, dtype=float)
probe = 1.0 / (indices ** 1.25)
probe = probe / np.sqrt(np.sum((1.0 / (np.arange(1, 5000) ** 1.25)) ** 2))
compact_image_norm = 1.0 / indices
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot(indices, strong_norms, color=PALETTE["red"], linewidth=2, label="strong norm ||e_n||")
ax.plot(indices, np.abs(probe), color=PALETTE["blue"], linewidth=2, label="fixed probe |<e_n,w>|")
ax.plot(indices, compact_image_norm, color=PALETTE["green"], linewidth=2, label="compact image ||K e_n||")
ax.set_yscale("log")
style_axis(ax, "Weak probes vanish while strong norms do not", xlabel="basis index n", ylabel="diagnostic magnitude")
ax.legend(loc="upper right")
add_note(ax, "K is diagonal with K e_n = (1/n)e_n")
fig.tight_layout()
weak_compact_path = remember(save_matplotlib(fig, TOPIC, "weak-strong-compact-probes.png", root=BOOK_ROOT / "artifacts"))
close(fig)
checks["weak_strong_compact"] = {
    "strong_norm_last": float(strong_norms[-1]),
    "probe_last": float(abs(probe[-1])),
    "probe_decreased": bool(abs(probe[-1]) < abs(probe[0])),
    "compact_image_norm_last": float(compact_image_norm[-1]),
    "compact_image_decreased": bool(compact_image_norm[-1] < compact_image_norm[0]),
}
display_artifact(weak_compact_path, width=760)
checks["weak_strong_compact"]

## 6. Functionals As Scalar Probes

A functional sends a vector or function to a scalar. In a Hilbert space, an important mental model is a Riesz-style probe: choose a representing function `g`, then define `phi_g(f) = <f,g>`. This is a scalar measurement of how much `f` aligns with `g`.

The pointwise product explains the integral visually. The coefficient bars show that the functional itself has coordinates: probing each basis vector recovers the coordinates of the representer `g`. The checks assert linearity and the Cauchy-Schwarz bound `|<f,g>| <= ||f|| ||g||`.

In [ ]:
g_raw = 0.90 + 0.55 * np.cos(2 * np.pi * x_grid) - 0.35 * np.sin(4 * np.pi * x_grid)
g_norm = np.sqrt(np.sum(g_raw * g_raw) * dx)
g = g_raw / g_norm
h = -0.25 + 0.30 * np.sin(4 * np.pi * x_grid) + 0.10 * np.cos(6 * np.pi * x_grid)
alpha, beta = 1.7, -0.4

def inner(a, b):
    return float(np.sum(a * b) * dx)

def l2norm(a):
    return float(np.sqrt(inner(a, a)))

phi_f = inner(f, g)
phi_h = inner(h, g)
linearity_residual = abs(inner(alpha * f + beta * h, g) - (alpha * phi_f + beta * phi_h))
cauchy_gap = l2norm(f) * l2norm(g) - abs(phi_f)
representer_coefficients = B[:9] @ g * dx
fig, axes = plt.subplots(3, 1, figsize=(9, 8), height_ratios=[1.1, 1.1, 1.0])
axes[0].plot(x_grid, f, color=PALETTE["blue"], linewidth=2, label="f")
axes[0].plot(x_grid, g, color=PALETTE["green"], linewidth=2, label="representer g")
style_axis(axes[0], "A functional can be represented by an inner-product probe", xlabel="x", ylabel="value")
axes[0].legend(fontsize=8, loc="best")
product = f * g
axes[1].plot(x_grid, product, color=PALETTE["violet"], linewidth=2)
axes[1].fill_between(x_grid, 0, product, where=product >= 0, color=PALETTE["violet"], alpha=0.20)
axes[1].fill_between(x_grid, 0, product, where=product < 0, color=PALETTE["red"], alpha=0.18)
style_axis(axes[1], f"pointwise product integrates to phi_g(f) = {phi_f:.3f}", xlabel="x", ylabel="f(x) g(x)")
axes[2].bar(np.arange(len(representer_coefficients)), representer_coefficients, color=PALETTE["gold"])
axes[2].set_xticks(np.arange(len(representer_coefficients)))
axes[2].set_xticklabels(basis_names[:9], rotation=35, ha="right", fontsize=8)
style_axis(axes[2], "Functional coordinates are coordinates of g", ylabel="phi_g(basis_i)")
fig.tight_layout()
functional_path = remember(save_matplotlib(fig, TOPIC, "functional-riesz-representer-probe.png", root=BOOK_ROOT / "artifacts"))
close(fig)
checks["functionals"] = {
    "phi_f": float(phi_f),
    "linearity_residual": float(linearity_residual),
    "cauchy_schwarz_gap": float(cauchy_gap),
    "representer_norm": float(l2norm(g)),
}
display_artifact(functional_path, width=760)
checks["functionals"]

## 7. Proof And Dependency Scaffold

Functional analysis definitions are easy to memorize in a disconnected way. The dependency graph below is a proof-reading scaffold: each edge means that the target concept uses the source concept in the version of the story developed here. Hilbert space needs both completeness and inner-product geometry; Fourier-style reconstruction needs orthonormal basis coordinates inside a Hilbert norm; adjoints and dual functionals use the same inner product to move between vectors and scalar probes.

In [ ]:
G = nx.DiGraph()
edges = [
    ("normed vector space", "Cauchy sequence"), ("Cauchy sequence", "completeness"), ("completeness", "Banach space"),
    ("inner product", "induced norm"), ("induced norm", "Hilbert space"), ("completeness", "Hilbert space"),
    ("Hilbert space", "orthogonal projection"), ("Hilbert space", "orthonormal basis"), ("orthonormal basis", "Fourier coefficients"), ("Fourier coefficients", "L2 reconstruction"),
    ("operator", "boundedness"), ("operator", "adjoint"), ("inner product", "adjoint"), ("adjoint", "self-adjoint operator"),
    ("weak convergence", "compact operator"), ("operator", "compact operator"), ("Hilbert space", "functional"),
    ("inner product", "Riesz-style probe"), ("functional", "dual space"), ("Riesz-style probe", "dual space"),
]
G.add_edges_from(edges)
pos = {
    "normed vector space": (0, 4), "Cauchy sequence": (1.6, 4), "completeness": (3.2, 4), "Banach space": (4.8, 4.5),
    "inner product": (0, 2.5), "induced norm": (1.6, 2.7), "Hilbert space": (3.2, 2.8), "orthogonal projection": (4.8, 3.2),
    "orthonormal basis": (4.8, 2.4), "Fourier coefficients": (6.4, 2.4), "L2 reconstruction": (8.0, 2.4),
    "operator": (3.2, 1.0), "boundedness": (4.8, 1.4), "adjoint": (4.8, 0.8), "self-adjoint operator": (6.4, 0.8),
    "weak convergence": (0, 0.4), "compact operator": (1.8, 0.4), "functional": (4.8, 0.0), "Riesz-style probe": (6.4, 0.0), "dual space": (8.0, 0.0),
}
fig, ax = plt.subplots(figsize=(12, 6))
node_colors = ["#dbeafe" if node in {"Banach space", "Hilbert space"} else "#dcfce7" if node in {"L2 reconstruction", "compact operator", "dual space"} else "#f8fafc" for node in G.nodes]
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=14, edge_color="#64748b", width=1.25)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, edgecolors="#334155", linewidths=1.0, node_size=1900)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8, font_color=PALETTE["ink"])
ax.set_title("Functional analysis dependency scaffold", color=PALETTE["ink"], fontsize=13)
ax.axis("off")
fig.tight_layout()
dependency_graph_path = remember(save_matplotlib(fig, TOPIC, "functional-analysis-dependency-graph.png", root=BOOK_ROOT / "artifacts"))
close(fig)
checks["dependency_graph"] = {
    "node_count": int(G.number_of_nodes()),
    "edge_count": int(G.number_of_edges()),
    "is_dag": bool(nx.is_directed_acyclic_graph(G)),
    "has_hilbert_to_reconstruction_path": bool(nx.has_path(G, "Hilbert space", "L2 reconstruction")),
    "has_functional_to_dual_path": bool(nx.has_path(G, "functional", "dual space")),
}
display_artifact(dependency_graph_path, width=900)
checks["dependency_graph"]

## 8. Applied Lab: Orthogonal Projection As Denoising

A common geometric-deep-learning move is to choose a basis that separates coarse structure from noisy high-frequency variation, then keep only part of the expansion. In a graph or manifold chapter, the basis might come from a Laplacian. Here we use the sampled orthonormal basis from the `L^2` section so the mechanism is transparent.

The lab adds deterministic noise to the clean function and projects the noisy signal onto low-dimensional spans. The projection is not magic: it helps when the clean signal is low-frequency relative to the noise, and the error table records that assumption numerically.

In [ ]:
rng = np.random.default_rng(42)
clean_signal = f
noise = 0.32 * rng.normal(size=sample_count)
noisy_signal = clean_signal + noise
noisy_coefficients = B @ noisy_signal * dx
lab_counts = [1, 3, 5, 7, 9, 13]
lab_rows = []
lab_recons = {}
clean_error_noisy = float(np.sqrt(np.sum((noisy_signal - clean_signal) ** 2) * dx))
for count in lab_counts:
    projected = noisy_coefficients[:count] @ B[:count]
    lab_recons[count] = projected
    lab_rows.append({
        "basis_terms": count,
        "l2_error_to_clean": float(np.sqrt(np.sum((projected - clean_signal) ** 2) * dx)),
        "retained_energy_fraction_of_noisy": float(np.sum(noisy_coefficients[:count] ** 2) / (np.sum(noisy_signal * noisy_signal) * dx)),
    })
lab_table_path = remember(save_table_csv(lab_rows, TOPIC, "projection-denoising-errors.csv", root=BOOK_ROOT / "artifacts"))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(x_grid, noisy_signal, color="#94a3b8", linewidth=1.0, label="noisy signal")
axes[0].plot(x_grid, clean_signal, color=PALETTE["ink"], linewidth=2.0, label="clean signal")
for count, color in [(3, PALETTE["gold"]), (7, PALETTE["teal"]), (13, PALETTE["blue"] )]:
    axes[0].plot(x_grid, lab_recons[count], linewidth=1.8, color=color, label=f"projection {count}")
style_axis(axes[0], "Projection keeps low-dimensional Hilbert structure", xlabel="x", ylabel="value")
axes[0].legend(fontsize=7, loc="best")
axes[1].axhline(clean_error_noisy, color=PALETTE["red"], linestyle="--", label="raw noisy error")
axes[1].plot([row["basis_terms"] for row in lab_rows], [row["l2_error_to_clean"] for row in lab_rows], "o-", color=PALETTE["blue"], label="projected error")
style_axis(axes[1], "Denoising works when the span matches the signal", xlabel="basis terms kept", ylabel="L2 error to clean")
axes[1].legend(fontsize=8, loc="best")
fig.tight_layout()
lab_path = remember(save_matplotlib(fig, TOPIC, "orthogonal-projection-denoising-lab.png", root=BOOK_ROOT / "artifacts"))
close(fig)
best_projected_error = min(row["l2_error_to_clean"] for row in lab_rows)
checks["applied_lab_projection"] = {
    "raw_noisy_l2_error": float(clean_error_noisy),
    "best_projected_l2_error": float(best_projected_error),
    "projection_improves_error": bool(best_projected_error < clean_error_noisy),
    "best_basis_terms": int(min(lab_rows, key=lambda row: row["l2_error_to_clean"])["basis_terms"]),
}
display_artifact(lab_path, width=860)
display_artifact(lab_table_path)
pd.DataFrame(lab_rows)

## Sanity checks

The final cell checks three layers:

1. **Artifact integrity:** every generated artifact exists and is nonempty; PNG files are not blank.
2. **Core identities:** Cauchy diagnostics, Hilbert projection, orthonormal Gram matrix, Parseval identity, adjoint identity, compact-probe behavior, functional boundedness, and graph structure all satisfy numerical or symbolic checks.
3. **Audit output:** a JSON summary is written to the chapter checks folder so later QC can inspect the exact numbers without re-running every plot manually.

In [ ]:
assert checks["cauchy_completeness"]["boundary_tail_diameter_last10"] < 0.002
assert checks["cauchy_completeness"]["closed_right_interval_contains_limit"] is True
assert checks["cauchy_completeness"]["open_interval_contains_limit"] is False
assert checks["cauchy_completeness"]["sqrt2_residual_last"] < 1e-12
assert checks["banach_hilbert_geometry"]["parallelogram_l2_residual"] < 1e-12
assert checks["banach_hilbert_geometry"]["parallelogram_l1_residual"] > 0.1
assert checks["banach_hilbert_geometry"]["projection_residual_dot_subspace"] < 1e-12
assert checks["orthonormal_reconstruction"]["gram_max_abs_error"] < 1e-12
assert checks["orthonormal_reconstruction"]["parseval_residual"] < 1e-12
assert checks["orthonormal_reconstruction"]["full_reconstruction_l2_error"] < 1e-12
assert checks["orthonormal_reconstruction"]["first_reconstruction_l2_error"] > checks["orthonormal_reconstruction"]["full_reconstruction_l2_error"]
assert checks["operators_adjoint"]["rotation_length_residual"] < 1e-12
assert checks["operators_adjoint"]["projection_idempotent_residual"] < 1e-12
assert checks["operators_adjoint"]["projection_self_adjoint_residual"] < 1e-12
assert checks["operators_adjoint"]["shear_self_adjoint_residual"] > 0.1
assert checks["operators_adjoint"]["numeric_adjoint_max_residual"] < 1e-12
assert checks["operators_adjoint"]["symbolic_adjoint_identity_zero"] is True
assert checks["weak_strong_compact"]["strong_norm_last"] == 1.0
assert checks["weak_strong_compact"]["probe_decreased"] is True
assert checks["weak_strong_compact"]["compact_image_decreased"] is True
assert checks["weak_strong_compact"]["compact_image_norm_last"] < 0.02
assert checks["functionals"]["linearity_residual"] < 1e-12
assert checks["functionals"]["cauchy_schwarz_gap"] >= -1e-12
assert abs(checks["functionals"]["representer_norm"] - 1.0) < 1e-12
assert checks["dependency_graph"]["is_dag"] is True
assert checks["dependency_graph"]["has_hilbert_to_reconstruction_path"] is True
assert checks["dependency_graph"]["has_functional_to_dual_path"] is True
assert checks["applied_lab_projection"]["projection_improves_error"] is True

invariants_path = remember(save_json(checks, TOPIC, "functional-analysis-invariants.json", root=BOOK_ROOT / "artifacts"))
records_before_final = assert_chapter_artifacts(artifact_paths)
image_records = [assert_nonblank_image(path) for path in artifact_paths if path.suffix.lower() == ".png"]
final_sanity = {
    "source_span": "PDF pages 51-54",
    "artifact_count_before_final": len(artifact_paths),
    "artifacts": [str(path.relative_to(BOOK_ROOT)).replace("\\", "/") for path in artifact_paths],
    "nonblank_png_count": len(image_records),
    "checks": checks,
}
final_sanity_path = remember(save_json(final_sanity, TOPIC, "final-sanity.json", root=BOOK_ROOT / "artifacts"))
records = assert_chapter_artifacts(artifact_paths)
assert len(records) == len(artifact_paths)
print(f"Validated {len(records)} artifacts, including {len(image_records)} nonblank PNG files.")
print(f"Invariant summary: {invariants_path.relative_to(BOOK_ROOT)}")
print(f"Final sanity summary: {final_sanity_path.relative_to(BOOK_ROOT)}")
display_artifact(invariants_path)
display_artifact(final_sanity_path)

## Takeaways

- A Cauchy tail is not enough; completeness decides whether the limit is inside the space.
- Banach spaces supply complete norm geometry; Hilbert spaces add inner products, orthogonality, and projection.
- Treating functions as vectors means choosing an inner product and a basis; coefficients are projections, not arbitrary labels.
- Operators become analyzable through boundedness, adjoints, isometries, projections, and compactness diagnostics.
- Functionals are scalar probes. In Hilbert settings, inner products with representing functions are the computational model to keep in mind.
- These ideas prepare the next chapter: spectral theory needs Hilbert geometry because eigenfunctions, self-adjoint operators, Fourier coefficients, and Laplacians all rely on the same inner-product structure.